# Explanation maps of the joint graders: a small localisation check and a few SHAP figures — record §100

**What this measures.** For both joint arms (direct and cropped; three seeds each, maps averaged over the seeds) three maps
of the expected grade — Grad-CAM, Integrated Gradients, Occlusion — are compared with the union of the expert
lesion masks (MA, HE, EX, SE) on a fixed sample of DDR lesion-segmentation **training** images, none of which is in the DDR
grading test list. Measures: pointing game, lesion share of the map's top 10 %, AUROC of the map as a lesion score, each
beside what a uniform map would score (the lesion share of the image; AUROC 0.5).

**What it does not show.** Overlap with lesions describes where the score is sensitive; it does not show that the grader
detects lesions or reasons as a clinician would. Nothing here selects a model, a method or a setting.

- SHAP (superpixel Kernel SHAP) runs on a few APTOS validation images only, for figures — never on a whole dataset.
- Uncertainty (seed ensemble, test-time augmentation) is not computed here; it belongs to the re-test notebook.
- Not read: the EyePACS held-out test; DDR or IDRiD grading test images.

Time on a T4: about 15 minutes per arm for 60 images (occlusion on 20), about 5 minutes for four SHAP figures.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. 
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import shutil, tarfile
import ddr_grading_data as dd, ddr_grading_eval as ev
import joint_train as jt, final_external_eval as fx, uncertainty_eval as ue, gradcam, demo_pipeline as demo, explain
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SETTINGS (fixed before running; nothing is chosen from the output) ====
RUN_LOCALISATION = False
N_IMAGES = 60              # DDR lesion-segmentation training images (fixed random sample, seed 20261014)
OCCLUSION_IMAGES = 20      # occlusion is the costly method: the first 20 of the sample
RUN_SHAP_FIGURES = False
SHAP_IMAGES = 4            # APTOS validation images, one each of grades 1-4 (the first in validation order)
print({"RUN_LOCALISATION": RUN_LOCALISATION, "N_IMAGES": N_IMAGES, "RUN_SHAP_FIGURES": RUN_SHAP_FIGURES})

In [ ]:
# ==== [3] THE SIX JOINT CHECKPOINTS (hash-checked against the frozen re-test protocol) ====
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
PROTOCOL = json.load(open(fx.PROTOCOL_PATH, encoding="utf-8"))
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
tf.keras.backend.clear_session()
_, ARRAYS = pl.load_reference(PRETRAINED)
EXPLAINERS = {}
for _model, _arm in fx.ARMS.items():
    EXPLAINERS[_arm] = []
    for _s in jt.SEEDS:
        _entry = PROTOCOL["checkpoints"][f"{_model}_seed{_s}"]
        _path = posixpath.join(DRIVE_ROOT, *_entry["path"].split("/"))
        assert fx.ig.sha256_file(_path) == _entry["sha256"], f"{_model} seed {_s}: not the pinned checkpoint"
        EXPLAINERS[_arm].append(gradcam.load_explainer(_s, _path, ARRAYS)[0])
FNS = {arm: [explain.graph_model_fn(g) for g in gs] for arm, gs in EXPLAINERS.items()}
print("explainers:", {arm: len(gs) for arm, gs in EXPLAINERS.items()})

In [ ]:
# ==== [4] LOCALISATION CHECK on DDR lesion-segmentation TRAINING images ====
OUT = posixpath.join(EXP, "FinalExternal", "explanations")
os.makedirs(OUT, exist_ok=True)
if RUN_LOCALISATION:
    LESION_ROOT = posixpath.join(colab_config.DATASET_ROOT, "DDR", "raw", "lesion_segmentation")
    ITEMS = explain.localisation_sample(LESION_ROOT, posixpath.join(colab_config.DATASET_ROOT, "DDR", "audit"), N_IMAGES)
    ROWS = []
    for _arm in ("direct", "crop"):
        ROWS += explain.localisation_study(ITEMS, _arm, EXPLAINERS[_arm], FNS[_arm],
                                           methods=("grad_cam", "integrated_gradients", "occlusion"), occlusion_images=OCCLUSION_IMAGES)
    LOC = pd.DataFrame(ROWS)
    LOC.to_csv(posixpath.join(OUT, "localisation_per_image.csv"), index=False)
    SUMMARY = LOC.groupby(["arm", "method"]).agg(images=("image", "count"), pointing_hit=("pointing_hit", "mean"), top10_precision=("top_precision", "mean"),
                                                 auroc=("auroc", "mean"), uniform_map_baseline=("lesion_fraction", "mean")).round(3)
    SUMMARY.to_csv(posixpath.join(OUT, "localisation_summary.csv"))
    display(SUMMARY)
    print("uniform-map baseline: pointing game and top-10 % precision = the lesion share of the image; AUROC = 0.5")
else:
    print("localisation not run (RUN_LOCALISATION = False)")

In [ ]:
# ==== [5] SHAP FIGURES (a few APTOS validation images only) ====
if RUN_SHAP_FIGURES:
    import matplotlib.pyplot as plt
    import pathology_grader_severity_fusion as sf
    import geometry_diagnostic as gd
    _ids, _grades = sf.authoritative_validation_ids()
    _pick = [next(i for i, g in zip(_ids, _grades) if g == grade) for grade in (1, 2, 3, 4)][:SHAP_IMAGES]
    for _id in _pick:
        _direct, _crop, _, _ = gd.two_frames(posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images", f"{_id}.png"))
        _fig, _axes = plt.subplots(1, 4, figsize=(18, 5))
        for _col, (_arm, _rgb) in enumerate((("direct", _direct), ("crop", _crop))):
            _heat = np.mean([explain.superpixel_shap(f, _rgb, n_segments=50, n_samples=512)[0] for f in FNS[_arm]], axis=0)
            _lim = float(np.abs(_heat).max()) or 1.0
            _axes[2 * _col].imshow(_rgb); _axes[2 * _col].set_title(f"{_id} ({_arm} frame)")
            _axes[2 * _col + 1].imshow(_heat, cmap="RdBu_r", vmin=-_lim, vmax=_lim); _axes[2 * _col + 1].set_title(f"SHAP of the expected grade ({_arm})")
        for _ax in _axes:
            _ax.axis("off")
        _fig.suptitle("Superpixel Kernel SHAP, three-seed mean. Red raised the expected grade, blue lowered it. Not evidence of lesion detection.")
        _fig.tight_layout(); _fig.savefig(posixpath.join(OUT, f"shap_{_id}.png"), dpi=110); plt.show()
else:
    print("SHAP figures not made (RUN_SHAP_FIGURES = False)")